# Perovskite Solar Cell Data — Exploration & Re-schema

โปรเจกต์นี้เน้น **กระบวนการ/ขั้นตอนการทดลอง (process)** ในการทำ perovskite solar cell
(ต่างจากงานที่เน้นค้นหาวัสดุ/ธาตุใหม่). Raw data อยู่ในโฟลเดอร์ `Raw Data/` โดย
**1 ไฟล์ `.xlsx` = 1 device (sample)** ตั้งชื่อว่า `<doi>_Sample<N>.xlsx`.

แต่ละ workbook มี 6 sheets: `Layer`, `Vender`, `Materials`, `Process`, `Properties`, `Stability`.

Notebook นี้แบ่งเป็น 3 ส่วน:
1. **Explore** — สำรวจไฟล์, ตรวจความสม่ำเสมอของ schema, หาปัญหา data quality
2. **Re-schema** — ออกแบบ + รัน ETL เปลี่ยนเป็นตาราง normalized ที่เชื่อมด้วย `device_id`
3. **Outputs & Queries** — เขียนออกเป็น CSV / SQLite / JSON และตัวอย่าง query

> สรุปคำแนะนำเรื่องรูปแบบจัดเก็บ (relational / nosql / csv / json / graphql) อยู่ใน **`RESCHEMA_README.md`**.

## 0. Setup

In [1]:
import sys, os, glob, re
import pandas as pd

# ทำให้ import scripts/reschema.py ได้ ไม่ว่าจะรัน notebook จากที่ไหน
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".." if os.path.basename(os.getcwd())=="notebooks" else "."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)

RAW_DIR = "Raw Data"
files = sorted(glob.glob(os.path.join(RAW_DIR, "*.xlsx")))
print("repo root :", ROOT)
print("workbooks :", len(files))

repo root : /Users/farlos/University/Perovskites-Research
workbooks : 44


## 1. Inventory — ไฟล์ / publication / sample

ชื่อไฟล์เข้ารหัส DOI + หมายเลข sample. แยกออกมาเป็นคีย์ที่ใช้งานได้.

In [2]:
from scripts.reschema import parse_device_id

inv = pd.DataFrame(
    [dict(zip(["device_id","doi_raw","sample_no"], parse_device_id(f))) | {"file": os.path.basename(f)}
     for f in files]
)
print(f"{len(inv)} devices จาก {inv['doi_raw'].nunique()} publications\n")
display(inv.head(10))
display(inv.groupby("doi_raw").size().rename("n_samples").sort_values(ascending=False))

44 devices จาก 15 publications



,device_id,doi_raw,sample_no,file
0,10.1002admi.202500231_Sample1,10.1002admi.202500231,1,10.1002admi.202500231_Sample1.xlsx
1,10.1002admi.202500231_Sample2,10.1002admi.202500231,2,10.1002admi.202500231_Sample2.xlsx
2,10.1002admi.202500231_Sample3,10.1002admi.202500231,3,10.1002admi.202500231_Sample3.xlsx
3,10.1002admi.202500231_Sample4,10.1002admi.202500231,4,10.1002admi.202500231_Sample4.xlsx
4,10.1016j.jechem.2021.11.029_Sample1,10.1016j.jechem.2021.11.029,1,10.1016j.jechem.2021.11.029_Sample1.xlsx
5,10.1016j.jechem.2021.11.029_Sample2,10.1016j.jechem.2021.11.029,2,10.1016j.jechem.2021.11.029_Sample2.xlsx
6,10.1016j.nanoen.2021.106285_Sample1,10.1016j.nanoen.2021.106285,1,10.1016j.nanoen.2021.106285_Sample1.xlsx
7,10.1016j.nanoen.2021.106285_Sample2,10.1016j.nanoen.2021.106285,2,10.1016j.nanoen.2021.106285_Sample2.xlsx
8,10.1016j.nanoen.2021.106285_Sample3,10.1016j.nanoen.2021.106285,3,10.1016j.nanoen.2021.106285_Sample3.xlsx
9,10.1021_acsami.5c16972_Sample1,10.1021_acsami.5c16972,1,10.1021_acsami.5c16972_Sample1.xlsx


doi_raw
10_1002adma_202507730             5
10_1016_j_heliyon_2024_e25352     5
10.1002admi.202500231             4
10_1039_d3lf00258f                4
10.1016j.nanoen.2021.106285       3
10.1021_acsami.5c16972            3
10_1021_acsami_5c15868            3
10_31635_ccschem_025_202505977    3
10.1016j.jechem.2021.11.029       2
10_1002_smll_202504237            2
10_1002_solr_202200422            2
10_1002_solr_202400534            2
10_1016_j_solener_2025_113286     2
10_1021_acsami_4c14300            2
10_1126_science_ade9637           2
Name: n_samples, dtype: int64

## 2. Schema audit — sheet และ header สม่ำเสมอแค่ไหน?

ตรวจว่าทุกไฟล์มี sheet/หัวคอลัมน์ตรงกันหรือไม่ (จุดที่ต้อง normalize).

In [3]:
from collections import Counter

sheet_combos, headers = Counter(), {}
for f in files:
    xl = pd.ExcelFile(f)
    sheet_combos[tuple(xl.sheet_names)] += 1
    for sh in xl.sheet_names:
        h = tuple(str(x).strip() for x in pd.read_excel(f, sh, nrows=0).columns)
        headers.setdefault(sh, Counter())[h] += 1

print("=== ชุดชื่อ sheet ที่พบ ===")
for combo, n in sheet_combos.most_common():
    print(f"  {n:2d}x  {combo}")

print("\n=== จำนวน header แบบที่ต่างกัน ต่อ sheet ===")
for sh, c in headers.items():
    print(f"  {sh:12s}: {len(c)} แบบ")

=== ชุดชื่อ sheet ที่พบ ===
  35x  ('Layer', 'Vender', 'Materials', 'Process', 'Properties', 'Stability')
   9x  ('Layer', 'Materials', 'Vender', 'Process', 'Properties', 'Stability')

=== จำนวน header แบบที่ต่างกัน ต่อ sheet ===
  Layer       : 1 แบบ
  Vender      : 1 แบบ
  Materials   : 1 แบบ
  Process     : 1 แบบ
  Properties  : 2 แบบ
  Stability   : 7 แบบ


In [4]:
# sheet ที่ header ไม่นิ่ง = Stability (และ Properties) -> ต้อง map ให้เป็นมาตรฐานเดียว
for h, n in headers["Stability"].most_common():
    print(f"{n:2d}x {h}")

27x ('Environment', 'Encapsulation', 'Temperature (°C)', 'Humidity (%)', 'Time (hr)', 'Retention (%)', 'T80 (hr)')
 6x ('Environment', 'Encapsulation', 'temperature (celsius)', 'relative humidity (%)', 'time (hr)', 'Normalize PCE (%)')
 4x ('Environment', 'Encapsulation', 'temperature (C)', 'humidity (%)', 'time (hr)', 'Retention (%)')
 4x ('Environment', 'Encapsulation', 'Temperature (°C)', 'humidity (%)', 'time (hr)', 'Retention (%)')
 1x ('Environment', 'Encapsulation', 'temperature (C)', 'humidity (%)', 'time (hr)', 'Normalize PCE (%)')
 1x ('Environment', 'Encapsulation', 'temperature (celsius)', 'relative humidity (%)', 'time (hr)', 'Final normalize PCE (%)')
 1x ('Environment', 'Encapsulation', 'Temperature (°C)', 'Humidity (%)', 'Time (hr)', 'Retention (%)')


## 3. ดูข้อมูลจริงของ 1 device (ครบ 6 sheets)

In [5]:
sample_file = files[0]
print("FILE:", os.path.basename(sample_file), "\n")
for sh in ["Layer","Vender","Materials","Process","Properties","Stability"]:
    print(f"───── {sh} ─────")
    display(pd.read_excel(sample_file, sh))

FILE: 10.1002admi.202500231_Sample1.xlsx 

───── Layer ─────


,LayerID,LayerType,Material,Thickness (nm)
0,1,Anode,ITO,NaN
1,2,HTL,NiOx,NaN
2,3,HTL treatment,NaN,NaN
3,4,PSK,[ (Cs.05FA.79MA.16)Pb(I.87Br.13)3 ],NaN
4,5,PSK treatment,NaN,NaN
5,6,ETL,C60,50.0
6,7,ETL treatment,BCP,10.0
7,8,Cathode,Ag,100.0


───── Vender ─────


,Material,%Purity,Vender
0,ITO coated glass,9 - 15 Ω □-1,"Colorado Concept Coatings LLC, 96041"
1,DMF,99.9,Sigma-Aldrich
2,DMSO,99,Alfa Aesar
3,C60,99,MER Corporation
4,BCP,96,Sigma Aldrich
5,PbI2,99.99,TCI
6,CsI,99.999,Sigma Aldrich
7,MABr,>99.99,Greatcell Solar Materials
8,PbBr2,99.99,TCI
9,FAI,99.99,\nGreatcell Solar Materials


───── Materials ─────


,LayerID,Material,Type,Solvent,Amount,Unit
0,1,ITO,NaN,NaN,NaN,NaN
1,2,NiOx NPs,Solution,H2O,22.0,mg/ml
2,3,NaN,NaN,NaN,NaN,NaN
3,3,NaN,NaN,NaN,NaN,NaN
4,4,PbI2,Solution,DMF/DMSO (4/1 V/V),324.0,mg
5,4,CsI,Solution,DMF/DMSO (4/1 V/V),11.0,mg
6,4,PbBr2,Solution,DMF/DMSO (4/1 V/V),47.0,mg
7,4,MABr,solution,DMF/DMSO (4/1 V/V),14.0,NaN
8,4,FAI,Solution,DMF/DMSO (4/1 V/V),110.0,mg
9,5,NaN,NaN,NaN,NaN,NaN


... (2 outputs left out)

───── Properties ─────


,Properties,Values,Errors,Unit,Remarks
0,VBM,-5.600,NaN,eV,For ITO/NiOx
1,Voc,1.020,0.010,V,AM 1.5G
2,Jsc,20.970,1.130,mA cm−2,AM 1.5G
3,FF,76.000,0.005,%,AM 1.5G
4,PCE,16.310,1.000,%,AM 1.5G
5,Rs,0.680,NaN,Ohm/cm2,NaN
6,FWHM,0.144,0.079,eV,NaN
7,HTL optical Band gap,3.850,NaN,eV,NiOx NPs
8,PSK Optical band gap,1.609,NaN,eV,NaN
9,VBM (UPS),-5.720,NaN,eV,NaN


───── Stability ─────


,Environment,Encapsulation,temperature (celsius),relative humidity (%),time (hr),Normalize PCE (%)
0,ISOS-L2,N,NaN,NaN,72,NaN


## 4. Data-quality scan

ประเด็นที่เจอ และต้องจัดการตอน re-schema:
- ชื่อ `Properties` เป็น free-text หลากหลายมาก (200+ ชื่อ เช่น `Champion PCE`, `Avg PCE`, `PCE (RS)`)
  → เก็บแบบ **long / key–value (EAV)** ไม่ใช่ wide columns
- header ของ `Stability` สะกด/ใช้ตัวพิมพ์ต่างกัน → map เป็นชื่อมาตรฐาน
- มีเซลล์ว่าง, ช่องว่าง/ขึ้นบรรทัดปนในข้อความ, แถวว่าง → ต้อง clean

In [6]:
prop_names = Counter()
for f in files:
    s = pd.read_excel(f, "Properties")["Properties"].dropna().astype(str).str.strip()
    prop_names.update(s)
print(f"property names ที่ไม่ซ้ำกัน: {len(prop_names)} ชื่อ  (top 15)")
for name, n in prop_names.most_common(15):
    print(f"  {n:3d}  {name}")

property names ที่ไม่ซ้ำกัน: 138 ชื่อ  (top 15)
   26  VTFL
   24  Champion PCE
   20  Champion Voc
   20  Champion Jsc
   20  Champion FF
   15  EIS Rs
   15  EIS Rrec
   11  VBM
    9  Grain Size
    8  Average Voc
    8  Average Jsc
    8  Average FF
    8  Average PCE
    8  Trap density
    8  Contact Angle


## 5. Re-schema — โมเดล normalized

ทุกตารางเชื่อมกันด้วยคีย์เดียว **`device_id`** (= ชื่อไฟล์ เช่น `10_1126_science_ade9637_Sample2`).

```
publication (doi_raw)
   └─< devices (device_id)                     # 1 แถว / sample
         ├─< layers        (device_id, layer_id, layer_type, material, thickness_nm)
         ├─< materials      (device_id, layer_id, material, type, solvent, amount, unit)
         ├─< process_steps  (device_id, layer_id, step_order, process_type, description, parameter, unit)
         ├─< vendors        (device_id, material, purity, vendor)
         ├─< properties     (device_id, property, value, error, unit, remarks)   # long/EAV
         └─< stability      (device_id, environment, encapsulation, temperature_c, humidity_pct, time_hr, retention_pct, t80_hr)
```

`layers.layer_id` เป็น natural key ภายใน device ที่ `materials` และ `process_steps` อ้างถึง
→ ทำให้ตอบได้ว่า "วัสดุ/ขั้นตอนไหนอยู่ layer ไหน" ซึ่งเป็นหัวใจของโปรเจกต์ที่เน้น *process*.

ETL logic อยู่ใน [`scripts/reschema.py`](../scripts/reschema.py) (ใช้ซ้ำได้นอก notebook).

In [7]:
from importlib import reload
import scripts.reschema as rs
reload(rs)

tables = rs.build_tables()
for name, df in tables.items():
    print(f"{name:12s} {df.shape}")

devices      (44, 4)
layers       (352, 5)
vendors      (640, 4)
materials    (546, 7)
process      (697, 7)
properties   (596, 6)
stability    (62, 8)


/Users/farlos/University/Perovskites-Research/scripts/reschema.py:107: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


In [8]:
for name in ["devices","layers","materials","process","properties","stability"]:
    print(f"\n───── {name} ─────")
    display(tables[name].head(6))


───── devices ─────


,device_id,doi_raw,sample_no,source_file
0,10.1002admi.202500231_Sample1,10.1002admi.202500231,1,10.1002admi.202500231_Sample1.xlsx
1,10.1002admi.202500231_Sample2,10.1002admi.202500231,2,10.1002admi.202500231_Sample2.xlsx
2,10.1002admi.202500231_Sample3,10.1002admi.202500231,3,10.1002admi.202500231_Sample3.xlsx
3,10.1002admi.202500231_Sample4,10.1002admi.202500231,4,10.1002admi.202500231_Sample4.xlsx
4,10.1016j.jechem.2021.11.029_Sample1,10.1016j.jechem.2021.11.029,1,10.1016j.jechem.2021.11.029_Sample1.xlsx
5,10.1016j.jechem.2021.11.029_Sample2,10.1016j.jechem.2021.11.029,2,10.1016j.jechem.2021.11.029_Sample2.xlsx



───── layers ─────


,device_id,layer_id,layer_type,material,thickness_nm
0,10.1002admi.202500231_Sample1,1,Anode,ITO,NaN
1,10.1002admi.202500231_Sample1,2,HTL,NiOx,NaN
2,10.1002admi.202500231_Sample1,3,HTL treatment,<NA>,NaN
3,10.1002admi.202500231_Sample1,4,PSK,[ (Cs.05FA.79MA.16)Pb(I.87Br.13)3 ],NaN
4,10.1002admi.202500231_Sample1,5,PSK treatment,<NA>,NaN
5,10.1002admi.202500231_Sample1,6,ETL,C60,50.0



───── materials ─────


,device_id,layer_id,material,type,solvent,amount,unit
0,10.1002admi.202500231_Sample1,1.0,ITO,<NA>,<NA>,NaN,<NA>
1,10.1002admi.202500231_Sample1,2.0,NiOx NPs,Solution,H2O,22.0,mg/ml
2,10.1002admi.202500231_Sample1,3.0,<NA>,<NA>,<NA>,NaN,<NA>
3,10.1002admi.202500231_Sample1,3.0,<NA>,<NA>,<NA>,NaN,<NA>
4,10.1002admi.202500231_Sample1,4.0,PbI2,Solution,DMF/DMSO (4/1 V/V),324.0,mg
5,10.1002admi.202500231_Sample1,4.0,CsI,Solution,DMF/DMSO (4/1 V/V),11.0,mg


... (2 outputs left out)


───── properties ─────


,device_id,property,value,error,unit,remarks
0,10.1002admi.202500231_Sample1,VBM,-5.60,NaN,eV,For ITO/NiOx
1,10.1002admi.202500231_Sample1,Voc,1.02,0.010,V,AM 1.5G
2,10.1002admi.202500231_Sample1,Jsc,20.97,1.130,mA cm−2,AM 1.5G
3,10.1002admi.202500231_Sample1,FF,76.00,0.005,%,AM 1.5G
4,10.1002admi.202500231_Sample1,PCE,16.31,1.000,%,AM 1.5G
5,10.1002admi.202500231_Sample1,Rs,0.68,NaN,Ohm/cm2,<NA>



───── stability ─────


,device_id,environment,encapsulation,temperature_c,humidity_pct,time_hr,retention_pct,t80_hr
0,10.1002admi.202500231_Sample1,ISOS-L2,N,NaN,NaN,72.0,NaN,NaN
1,10.1002admi.202500231_Sample2,ISOS-L2,N,NaN,NaN,72.0,NaN,NaN
2,10.1002admi.202500231_Sample3,ISOS-L2,N,NaN,NaN,72.0,NaN,NaN
3,10.1002admi.202500231_Sample4,ISOS-L2,N,NaN,NaN,72.0,NaN,NaN
4,10.1016j.jechem.2021.11.029_Sample1,Nitrogen (N2),N,NaN,NaN,2880.0,86,NaN
5,10.1016j.jechem.2021.11.029_Sample1,Operational (MPP),N,NaN,NaN,700.0,66,NaN


## 6. ⭐ ตารางพร้อมใช้กับ ML/DL — `ml_dataset.csv` (1 แถว / device)

โมเดล normalized ด้านบนเหมาะเป็น **source of truth** แต่ ML/DL ต้องการ **feature matrix
1 แถวต่อ 1 sample**. `build_ml_table()` จึง pivot/aggregate ตาราง long กลับมาเป็นตารางกว้างตารางเดียว:

- **identifier**: `device_id`, `doi_raw`, `sample_no`
- **structural** (จาก layer stack): `n_layers`, `htl_material`, `etl_material`, `perovskite`, `anode`, `cathode`
- **process** (หัวใจของโปรเจกต์): `n_process_steps`, `n_spin_coating`, `has_antisolvent`
- **JV / target** (pivot จาก Properties, ใช้ค่า max≈champion): `pce_max`, `voc_max`, `jsc_max`, `ff_max`, `bandgap_max`

นี่คือรูปแบบที่ **แนะนำให้ใช้** — โหลดด้วย `pd.read_csv()` แล้วเข้า scikit-learn / PyTorch ได้ทันที.

In [9]:
ml = rs.build_ml_table(tables)
print("ML table:", ml.shape)
display(ml.head(8))
print("\nค่าที่ไม่ว่าง (non-null) ต่อคอลัมน์:")
print(ml.notna().sum())

ML table: (44, 17)


/Users/farlos/University/Perovskites-Research/scripts/reschema.py:228: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  feat["has_antisolvent"] = feat["has_antisolvent"].fillna(False)


,device_id,doi_raw,sample_no,n_layers,htl_material,etl_material,perovskite,anode,cathode,n_process_steps,n_spin_coating,has_antisolvent,bandgap_max,ff_max,jsc_max,pce_max,voc_max
0,10.1002admi.202500231_Sample1,10.1002admi.202500231,1,6,NiOx,C60,[ (Cs.05FA.79MA.16)Pb(I.87Br.13)3 ],ITO,Ag,14,3,True,3.85,76.00,20.97,16.31,1.020
1,10.1002admi.202500231_Sample2,10.1002admi.202500231,2,7,NiOx NPs,C60,[ (Cs0.05FA0.79MA0.16)Pb(I.87Br0.13)3 ],ITO,Ag,17,4,True,3.85,75.00,22.01,17.11,1.040
2,10.1002admi.202500231_Sample3,10.1002admi.202500231,3,7,NiOx NPs,C60,[ (Cs.05FA.79MA.16)Pb(I.87Br.13)3 ],ITO,Ag,17,4,True,3.85,78.00,21.86,16.97,1.000
3,10.1002admi.202500231_Sample4,10.1002admi.202500231,4,7,NiOx NPs,C60,[ (Cs0.05FA0.79MA0.16)Pb(I.87Br0.13)3 ],ITO,Ag,16,4,True,3.85,79.00,20.79,16.25,0.990
4,10.1016j.jechem.2021.11.029_Sample1,10.1016j.jechem.2021.11.029,1,7,NiOx,PCBM,(FA0.85MA0.15)0.95Cs0.05Pb(I0.85Br0.15)3,ITO,Ag,13,6,True,NaN,79.60,21.32,17.83,1.065
5,10.1016j.jechem.2021.11.029_Sample2,10.1016j.jechem.2021.11.029,2,8,NiOx,PCBM,(FA0.85MA0.15)0.95Cs0.05Pb(I0.85Br0.15)3,ITO,Ag,15,7,True,0.42,82.09,22.12,20.16,1.122
6,10.1016j.nanoen.2021.106285_Sample1,10.1016j.nanoen.2021.106285,1,8,NiOx,PCBM,(FA0.83MA0.17)0.95Pb(I0.9Br0.1)3,FTO,Ag,18,7,True,NaN,74.89,22.92,18.23,1.062
7,10.1016j.nanoen.2021.106285_Sample2,10.1016j.nanoen.2021.106285,2,8,NiOx,PCBM,Cs0.04FA0.83MA0.12Pb(I0.88Br0.12)3,FTO,Ag,19,7,True,NaN,76.69,23.95,20.28,1.104



ค่าที่ไม่ว่าง (non-null) ต่อคอลัมน์:
device_id          44
doi_raw            44
sample_no          44
n_layers           44
htl_material       44
etl_material       44
perovskite         44
anode              44
cathode            44
n_process_steps    44
n_spin_coating     44
has_antisolvent    44
bandgap_max         6
ff_max             43
jsc_max            43
pce_max            43
voc_max            43
dtype: int64


## 7. เขียน output (CSV เท่านั้น)

ตามที่สรุปไว้ — เก็บเป็น **CSV** อย่างเดียว: ไม่ต้องตั้ง server, version control ได้, โหลดเข้า ML ตรงๆ
- `processed/ml_dataset.csv` — ⭐ ตารางพร้อมเทรน (1 แถว/device)
- `processed/csv/*.csv` — ตาราง normalized (source of truth ไว้สร้าง feature เพิ่ม)

> ทางเลือกอื่น (SQLite / JSON) มีให้ใน `scripts/reschema.py` ผ่าน `write_sqlite()` / `write_json()`
> แต่**ไม่ใช่เส้นทางที่แนะนำ** สำหรับงาน ML/DL — ดูเหตุผลใน `RESCHEMA_README.md`.

In [10]:
ml_path = rs.write_ml_table(tables)
rs.write_csv(tables)
print("wrote ->", ml_path)
print("wrote ->", os.path.join(rs.OUT_DIR, "csv", "*.csv"))

wrote -> processed/ml_dataset.csv
wrote -> processed/csv/*.csv


/Users/farlos/University/Perovskites-Research/scripts/reschema.py:228: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  feat["has_antisolvent"] = feat["has_antisolvent"].fillna(False)


## 8. ตัวอย่างการใช้งานต่อ (pandas)

ทุกอย่างทำบน DataFrame ตรงๆ ไม่ต้องใช้ SQL/DB.

In [11]:
# Top devices ตาม PCE
display(ml.sort_values("pce_max", ascending=False)
          [["device_id","htl_material","perovskite","pce_max","voc_max","jsc_max","ff_max"]].head(10))

,device_id,htl_material,perovskite,pce_max,voc_max,jsc_max,ff_max
22,10_1002adma_202507730_Sample5,NiOx Nanoparticles,Cs0.05MA0.05FA0.90PbI3,26.50,1.195,26.51,83.60
21,10_1002adma_202507730_Sample4,NiOx Nanoparticles,Cs0.05MA0.05FA0.90PbI3,25.84,1.180,26.48,82.70
40,10_1126_science_ade9637_Sample2,NiOx NPs,Cs0.05FA0.85MA0.1PbI3,25.60,1.190,25.40,84.60
20,10_1002adma_202507730_Sample3,NiOx Nanoparticles,Cs0.05MA0.05FA0.90PbI3,25.36,1.171,26.55,81.60
43,10_31635_ccschem_025_202505977_Sample3,NiOx NPs,Cs0.05(FA0.95MA0.05)0.95Pb(I0.95Br0.05)3,25.28,1.200,25.48,82.71
18,10_1002adma_202507730_Sample1,NiOx Nanoparticles,Cs0.05MA0.05FA0.90PbI3,24.50,1.158,26.35,80.40
17,10_1002_solr_202400534_Sample2,NiOx NPs,Cs0.05(FA0.83MA0.12)0.95Pb(I0.83Br0.17)3 + TFM...,24.21,1.155,25.96,80.40
11,10.1021_acsami.5c16972_Sample3,NiOx,[(FA0.83MA0.17)0.95Cs0.05Pb(I0.9Br0.1)3],23.68,1.208,24.88,84.38
34,10_1021_acsami_5c15868_Sample3,NiOx,Cs0.11(FA0.83MA0.17)0.89Pb(Br0.15I0.85)3,23.31,1.240,22.95,82.91
42,10_31635_ccschem_025_202505977_Sample2,NiOx NPs,Cs0.05(FA0.95MA0.05)0.95Pb(I0.95Br0.05)3,23.07,1.190,24.29,79.77


In [12]:
# HTL material ที่ใช้บ่อย + PCE เฉลี่ย (เชื่อม process -> performance)
display(ml.groupby("htl_material")
          .agg(n=("device_id","size"), mean_pce=("pce_max","mean"))
          .sort_values("n", ascending=False))

,n,mean_pce
htl_material,,
NiOx,26,18.772308
NiOx NPs,11,20.977273
NiOx Nanoparticles,5,25.550000
Cu:NiO,1,16.610000
NiOx:TTTS NPs,1,22.810000


In [13]:
# โหลดเข้า ML ได้ทันที: แยก feature / target
import pandas as pd
df = pd.read_csv("processed/ml_dataset.csv")

target = "pce_max"
numeric_X = ["n_layers","n_process_steps","n_spin_coating","voc_max","jsc_max","ff_max"]
cat_X     = ["htl_material","etl_material"]

model_df = df.dropna(subset=[target])
print("ตัวอย่างtรียม X/y สำหรับ regression ทำนาย PCE:")
print("X numeric:", numeric_X)
print("X categorical (ต้อง encode):", cat_X)
print("y:", target, "| n_samples =", len(model_df))
display(model_df[[target] + numeric_X + cat_X].head())

ตัวอย่างtรียม X/y สำหรับ regression ทำนาย PCE:
X numeric: ['n_layers', 'n_process_steps', 'n_spin_coating', 'voc_max', 'jsc_max', 'ff_max']
X categorical (ต้อง encode): ['htl_material', 'etl_material']
y: pce_max | n_samples = 43


,pce_max,n_layers,n_process_steps,n_spin_coating,voc_max,jsc_max,ff_max,htl_material,etl_material
0,16.31,6,14,3,1.020,20.97,76.0,NiOx,C60
1,17.11,7,17,4,1.040,22.01,75.0,NiOx NPs,C60
2,16.97,7,17,4,1.000,21.86,78.0,NiOx NPs,C60
3,16.25,7,16,4,0.990,20.79,79.0,NiOx NPs,C60
4,17.83,7,13,6,1.065,21.32,79.6,NiOx,PCBM


---
### สรุป
- raw `.xlsx` (1 ไฟล์/device) → 7 ตาราง normalized (source of truth) → **`ml_dataset.csv`** (1 แถว/device, พร้อม ML)
- รูปแบบที่แนะนำ = **CSV** (ดูเหตุผลเทียบกับ relational/nosql/json/graphql ใน `RESCHEMA_README.md`)
- regenerate ทั้งหมด: `python scripts/reschema.py`
- ขยาย feature: แก้ `build_ml_table()` ใน `scripts/reschema.py`